# 02 - Generación de Datos

Generación de pares pregunta-SQL usando un LLM sobre los esquemas propios.

In [ ]:
import json
import os
from pathlib import Path
from tqdm import tqdm
import random
import re

## 1. Cargar esquemas propios

In [ ]:
def cargar_esquemas(ruta='datos/esquemas/entrenamiento/'):
    """Carga todos los esquemas SQL de la carpeta indicada."""
    esquemas = {}
    for archivo in Path(ruta).glob('*.sql'):
        nombre = archivo.stem
        with open(archivo, 'r') as f:
            esquemas[nombre] = f.read()
    return esquemas

esquemas = cargar_esquemas()
print(f'Esquemas cargados: {list(esquemas.keys())}')

## 2. Prompt para generación con LLM

Usamos un LLM para generar pares pregunta-SQL sobre cada esquema.

In [ ]:
PROMPT_GENERACION = """Dado el siguiente esquema de base de datos MySQL, genera {num_preguntas} pares de pregunta y consulta SQL.

Esquema:
{esquema}

Requisitos:
- Las preguntas deben estar en español
- Las consultas deben ser MySQL válidas (SELECT o WITH ... SELECT únicamente)
- Distribución: {distribucion}
- Las consultas deben ser correctas y ejecutables en MySQL 8.4.10
- No uses nombres de columnas o tablas que no existan en el esquema
- Incluye JOINs, GROUP BY, HAVING, subconsultas según el nivel

Responde en formato JSON:
[
  {{
    "pregunta": "...",
    "sql": "...",
    "nivel": "facil|media|dificil"
  }}
]
"""

DISTRIBUCION_NIVELES = {
    'facil': '40%: una tabla, filtros simples, ORDER BY, LIMIT',
    'media': '40%: JOINs, GROUP BY, HAVING, agregaciones',
    'dificil': '20%: subconsultas, WITH, funciones de ventana, fechas'
}

## 3. Función de generación

In [ ]:
def generar_pares(esquema, num_preguntas=20, modelo='local'):
    """
    Genera pares pregunta-SQL para un esquema dado.
    
    NOTA: Esta función requiere un LLM para generar los pares.
    Opciones:
    1. Usar un modelo local grande (Qwen2.5-Coder-7B-Instruct o similar)
    2. Usar API de un LLM (si se permite)
    3. Generar manualmente + templates
    """
    prompt = PROMPT_GENERACION.format(
        esquema=esquema,
        num_preguntas=num_preguntas,
        distribucion='; '.join(f"{k}: {v}" for k, v in DISTRIBUCION_NIVELES.items())
    )
    
    # TODO: Implementar generación con el LLM seleccionado
    # Por ahora, generar con templates básicos
    return generar_con_templates(esquema, num_preguntas)


def generar_con_templates(esquema, num_preguntas):
    """
    Generación de pares usando templates para las tablas del esquema.
    Extrae tablas y columnas del esquema y genera preguntas básicas.
    """
    # Extraer nombres de tablas
    tablas = re.findall(r'CREATE TABLE (\w+)', esquema, re.IGNORECASE)
    # Extraer columnas con sus comentarios
    columnas = {}
    for tabla in tablas:
        cols = re.findall(
            rf'{tabla}.*?(?=CREATE TABLE|$)',
            esquema, re.IGNORECASE | re.DOTALL
        )
        if cols:
            col_names = re.findall(r'(\w+)\s+\w+.*?COMMENT\s+\'(.+?)\'', cols[0])
            columnas[tabla] = [(c[0], c[1]) for c in col_names]
    
    pares = []
    # Templates para preguntas fáciles
    for tabla in tablas:
        cols_info = columnas.get(tabla, [])
        if cols_info:
            col_nombre = cols_info[0][0]
            pares.append({
                'pregunta': f'¿Cuántos registros hay en la tabla {tabla}?',
                'sql': f'SELECT COUNT(*) FROM {tabla}',
                'nivel': 'facil'
            })
            pares.append({
                'pregunta': f'Lista todos los {tabla}',
                'sql': f'SELECT * FROM {tabla}',
                'nivel': 'facil'
            })
    
    return pares[:num_preguntas]

## 4. Generar datos para todos los esquemas

In [ ]:
todos_los_pares = []

for nombre, esquema in esquemas.items():
    print(f'Generando para: {nombre}')
    pares = generar_pares(esquema, num_preguntas=200)
    todos_los_pares.extend(pares)
    print(f'  Generados: {len(pares)} pares')

print(f'\nTotal generado: {len(todos_los_pares)} pares')

## 5. Guardar dataset

In [ ]:
# Guardar dataset de entrenamiento
output_path = Path('datos/entrenamiento/dataset_completo.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(todos_los_pares, f, ensure_ascii=False, indent=2)

print(f'Dataset guardado en: {output_path}')
print(f'Total pares: {len(todos_los_pares)}')

## 6. Registrar descartes

In [ ]:
descartes = {
    'total_generados': len(todos_los_pares),
    'descartados_sintaxis': 0,
    'descartados_ejecucion': 0,
    'descartados_duplicados': 0,
    'validos': len(todos_los_pares),
    'razones': []
}

# TODO: Validar cada par en MySQL 8.4.10 y registrar descartes

with open('datos/descartes/reporte_descartes.json', 'w') as f:
    json.dump(descartes, f, ensure_ascii=False, indent=2)

print('Reporte de descartes guardado')